# 18 · Joint operator gating + readout in one loop

Classic symbolic regression separates *feature selection* from *fitting*. The
`omnibias.torch` joint operator regressor learns both at once: a differentiable
gate over a typed operator dictionary plus a readout head, trained end-to-end. On
a controlled law `y = 3 x₁² - 2 x₂x₃ + sin(x₄) + noise` it ranks the true
operators at the top and matches an exhaustive dictionary baseline.

Source: `examples/symbolic_discovery/joint_operator_regressor/`.

In [ ]:
import sys, tempfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
sys.path.insert(0, "..")
from _style import set_style, PRIMARY, ACCENT, GOOD
set_style()

from examples.symbolic_discovery.joint_operator_regressor.benchmark import evaluate_benchmark

res = evaluate_benchmark(Path(tempfile.mkdtemp()), seed=0)
print("hidden law:", res["hidden_law"])

## Accuracy vs baselines

Raw ridge on 4 inputs cannot represent the nonlinearity; the joint model reaches
the noise floor, on par with a full hand-built dictionary.

In [ ]:
models = res["models"]
order = sorted(models.items(), key=lambda kv: kv[1]["rmse"])
names = [k for k, _ in order]
rmses = [v["rmse"] for _, v in order]
colors = [GOOD if "joint" in n else PRIMARY for n in names]
fig, ax = plt.subplots(figsize=(8.0, 3.6))
ax.barh(names, rmses, color=colors)
ax.set_xscale("log"); ax.set_xlabel("test RMSE (log)")
ax.set_title("Joint operator regressor vs ridge / dictionary")
plt.tight_layout()
for k, v in order:
    print(f"  {k:<36} RMSE={v['rmse']:.4f}")

## Which operators did it select?

The learned gate importances rank the true operators (`x₁²`, `x₂x₃`, `sin x₄`)
at the top, with no separate selection pass.

In [ ]:
ops = res["selected_operators"][:8]
labels = [row["name"] for row in ops][::-1]
imp = [row["importance"] for row in ops][::-1]
fig, ax = plt.subplots(figsize=(7.2, 4.0))
truth = {"x1^2", "x2*x3", "sin(x4)"}
bar_colors = [ACCENT if lbl in truth else PRIMARY for lbl in labels]
ax.barh(labels, imp, color=bar_colors)
ax.set_xlabel("learned operator importance")
ax.set_title("Top gated operators (true terms in red)")
plt.tight_layout()

## Takeaway

A single differentiable objective recovers both *which* operators matter and
*how much* — a learnable, typed alternative to two-stage SINDy. The real-data
refinement loop (C-MAPSS RUL) lives in the same experiment package.